In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop keeps calling the model by using a `while` loop that continues to execute as long as the model returns a response that includes function calls.

Here is how the process works:

1.  **The Loop Structure:** The agent is wrapped in a `while True` loop. Inside this loop, the code sends the current message history (which includes the prompt, previous model outputs, and tool results) to the LLM.
2.  **Tracking Calls:** Within the loop, the code checks the response from the model. If the model returns a `function_call`, the code executes that function using a helper (`make_call`), appends the tool's output to the conversation history, and sets a flag (`has_function_calls`) to `True`.
3.  **Iteration and Continuation:** Because the updated message history now contains the tool's output, the loop continues to the next iteration, sending this new information back to the model so it can reason about the next step.
4.  **Stopping Condition:** The loop checks the `has_function_call

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Q1. First trace

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult, SimpleSpanProcessor

class CompactSpanExporter(SpanExporter):
    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            duration_ms = (span.end_time - span.start_time) / 1_000_000
            print(f"[{span.name}] {duration_ms:.1f}ms | "
                  f"input_tokens={attrs.get('input_tokens')} "
                  f"output_tokens={attrs.get('output_tokens')}")
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass

    def force_flush(self, timeout_millis=30000):
        return True
    
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(CompactSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from starter import rag_traced

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

[search] 2.4ms | input_tokens=None output_tokens=None
[llm] 999.9ms | input_tokens=7933 output_tokens=122
[rag] 1003.1ms | input_tokens=None output_tokens=None
The agentic loop keeps calling the model by using a `while True` loop that repeatedly sends the conversation history—including previous prompts, model outputs, and tool results—to the LLM. 

Within this loop, the code checks the model's response for any `function_call` items. If function calls are detected, the agent executes the tools, appends the tool results to the message history, and continues to the next iteration. The loop terminates when the model returns a response that does not contain any function calls, signaling that it has finished its task and is providing a final answer.


Q2. Capturing metrics as span attributes

In [3]:
from starter import rag_traced

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

[search] 5.3ms | input_tokens=None output_tokens=None
[llm] 1535.2ms | input_tokens=7933 output_tokens=253
[rag] 1541.3ms | input_tokens=None output_tokens=None
The agentic loop keeps calling the model by wrapping the API call inside a `while` loop. 

Here is how the process works:

1.  **Continuous Loop:** The code uses a `while True` loop that repeatedly sends the message history to the model.
2.  **Tracking Calls:** Within the loop, the code checks the model's response for any `function_call` items. A flag (e.g., `has_function_calls`) is set to `True` if a tool is requested.
3.  **Executing Tools:** If the model requests a function call, the code executes the corresponding tool and appends the result to the message history. 
4.  **Stopping Condition:** The loop checks the `has_function_calls` flag at the end of each iteration. If the model returns a response that does **not** contain any function calls, the `has_function_calls` flag remains `False`, and the `break` statement exits t

Q3. Span timing

099878 − 096146 = 0.003732 s 

728609 − 102454 = 1.626155 s 

3.732 ms + 1626.155 ms = 1630 ms

Q4. Saving traces to SQLite

In [4]:
from exporter import SQLiteSpanExporter

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [5]:
from starter import rag_traced

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

[search] 6.6ms | input_tokens=None output_tokens=None
[llm] 1398.5ms | input_tokens=7933 output_tokens=231
[rag] 1413.6ms | input_tokens=None output_tokens=None
The agentic loop keeps calling the model by wrapping the API call inside a `while` loop. 

Here is how the process works:

1.  **Iteration:** The loop repeatedly sends the current message history (which includes the user's prompt, previous model outputs, and any tool results) to the model.
2.  **Tool Execution:** The code checks the model's response for `function_call` items. If any are found, it executes the corresponding tool (e.g., `search`) and appends the tool's output to the message history.
3.  **Exit Condition:** The loop uses a flag (such as `has_function_calls`) to track if the model requested any tools. If the model returns a response without any function calls, it means the model has finished its task, and the loop terminates (breaks).

In short, the model decides when it needs to use a tool, and the code runs that 

Q5. Querying trace data

In [6]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("""
    SELECT
        name,
        SUM(end_time - start_time) AS total_duration_ns
    FROM spans
    WHERE name != 'rag'
    GROUP BY name
""", conn)

df["total_duration_ms"] = df["total_duration_ns"] / 1_000_000
print(df)

     name  total_duration_ns  total_duration_ms
0     llm         7536714423        7536.714423
1  search           21463906          21.463906


Q6. Token stability across runs

In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("""
    SELECT name, start_time, input_tokens, output_tokens
    FROM spans
    WHERE name = 'llm'
    ORDER BY start_time
""", conn)

print(df)
print(df["input_tokens"].describe())
print("range:", df["input_tokens"].max() - df["input_tokens"].min())

  name           start_time  input_tokens  output_tokens
0  llm  1784575557252832649          7933            242
1  llm  1784575617131809005          7933            241
2  llm  1784576127231246712          7933            289
3  llm  1784576188778352916          7933            276
4  llm  1784577046383757420          7933            231
count       5.0
mean     7933.0
std         0.0
min      7933.0
25%      7933.0
50%      7933.0
75%      7933.0
max      7933.0
Name: input_tokens, dtype: float64
range: 0
